In [1]:
import os
os.environ["OPENAI_API_KEY"] = 'DUMMY'
os.environ["OLLAMA_HOST"] = 'http://192.168.1.42:11434/v1' # or http://localhost:11434/v1
os.environ["OLLAMA_TIMEOUT"] = '1200'
os.environ["ACCURACY_THRESHOLD"] = '0.5'
os.environ["MAX_TOKENS"] = '5000'

In [2]:
import json
import pickle
from pageindex import *
# from pageindex.page_index_md import md_to_tree
from pageindex.utils import ConfigLoader, llm_acompletion, create_node_mapping
from pprint import pprint

In [3]:
import nest_asyncio
nest_asyncio.apply()
import asyncio

### Config

In [4]:
user_opt = {
    'model':"openai/gpt-oss:20b", # gpt-oss:20b qwen3:14b,
    'retrieve_model':"openai/gpt-oss:20b",
    'if_add_node_id':"yes",
    'if_add_node_summary':"yes",
    'if_add_doc_description':"no",
    'if_add_node_text':"yes",
}

opt = ConfigLoader().load({k: v for k, v in user_opt.items() if v is not None})

## Step 1: PageIndex Tree Generation

In [5]:
%%time
pdf_path = './data\\2077.pdf'
def start():
    toc_with_page_number = page_index_main(pdf_path, opt)
    print('Processing complete')
    return toc_with_page_number

toc_with_page_number = start()

Parsing PDF...
start find_toc_pages
response {
    "thinking": "The provided text contains headings such as \"Highlights\", \"Abstract\", \"Keywords\", and a single numbered section \"1. Introduction\", but it does not include a separate list that enumerates sections, subsections, or page numbers typical of a table of contents. Therefore, there is no table of contents present.",
    "toc_detected": "no"
}
response {
    "thinking": "The provided text consists of continuous prose discussing the paper's motivation, context, and contributions. It does not contain a structured list of sections, chapters, or headings that would constitute a table of contents. Therefore, no table of contents is present.",
    "toc_detected": "no"
}
response {
    "thinking": "The provided text contains section headings such as \"2. Literature review\" and \"2.1. Intersection of science fiction and technological foresight\", but these are part of the main body of the document rather than a separate list of se

#### 1.1 Save or restore TOC with page numbers

In [6]:
with open("data/2077_result_w_text_57.pkl", 'wb') as f:
    pickle.dump(toc_with_page_number, f)

In [7]:
with open("data/2077_result_w_text_63.pkl", 'rb') as f:
    toc_with_page_number = pickle.load(f)

#### 1.2 Get the generated PageIndex tree structure

In [8]:
utils.list_to_tree(toc_with_page_number['structure'])

[{'title': 'Introduction', 'start_index': 1, 'end_index': 3},
 {'title': 'Literature review', 'start_index': 3, 'end_index': 3},
 {'title': 'Methodology', 'start_index': 7, 'end_index': 7},
 {'title': 'Themes and technologies', 'start_index': 8, 'end_index': 9},
 {'title': 'Discussion', 'start_index': 16, 'end_index': 19},
 {'title': 'Conclusion', 'start_index': 19, 'end_index': 19},
 {'title': 'CRediT authorship contribution statement',
  'start_index': 19,
  'end_index': 19},
 {'title': 'Declaration of Generative AI and AI-assisted technologies in the writing process',
  'start_index': 19,
  'end_index': 19}]

In [9]:
# Remove document text chunks from Tree
tree_with_text = toc_with_page_number['structure'].copy()
tree_without_text = utils.remove_fields(tree_with_text, fields=['text'])
# pprint(json.dumps(tree_without_text, indent=2))

In [10]:
# check JSON
utils.print_json(toc_with_page_number, max_len=50, indent=2)

{
  "doc_name": "2077.pdf",
  "structure": [
    {
      "title": "Introduction",
      "node_id": "0000",
      "start_index": 1,
      "end_index": 3,
      "summary": "The paper examines how the video game Cyberpunk 20...",
      "text": "A Cyberpunk 2077 perspective on the prediction and..."
    },
    {
      "title": "Literature review",
      "node_id": "0001",
      "start_index": 3,
      "end_index": 3,
      "summary": "The partial document presents a literature review ...",
      "text": "contextualizing Cyberpunk 2077 within the broader ...",
      "nodes": [
        {
          "title": "Intersection of science fiction and technological ...",
          "node_id": "0002",
          "start_index": 3,
          "end_index": 4,
          "summary": "The partial document surveys how science fiction, ...",
          "text": "contextualizing Cyberpunk 2077 within the broader ..."
        },
        {
          "title": "Science fiction and technological advancements",
          

## Step 2: Reasoning-Based Retrieval with Tree Search

#### 2.1 Use LLM for tree search and identify nodes that might contain relevant context

In [11]:
# query = "What is about simulated reality in this game?"
query = "What model was used in preparing this article and why?"

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

In [12]:
tree_search_result = await llm_acompletion(model="openai/gpt-oss:20b", prompt=search_prompt)

#### 2.2 Print retrieved nodes and reasoning process

In [13]:
print(tree_search_result)

{
    "thinking": "The question asks which model was used to prepare the article and why. The conclusion section (node 0027) explicitly states that GPT‑4 was used for drafting and formatting. The same statement appears in the CRediT authorship contribution statement (node 0028) and the declaration of generative AI (node 0029). These nodes directly answer the question.",
    "node_list": ["0027", "0028", "0029"]
}


In [14]:
tree_search_result_json = json.loads(tree_search_result)
# tree_search_result_json

In [15]:
node_map = utils.create_node_mapping(tree_with_text)
# pprint(node_map)

In [16]:
print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}")


Retrieved Nodes:
Node ID: 0027
Node ID: 0028
Node ID: 0029


## Step 3: Answer Generation

#### 3.1 Extract relevant context from retrieved nodes

In [17]:
node_list = json.loads(tree_search_result)["node_list"]
relevant_content = "\n\n".join(node_map[node_id]["text"] for node_id in node_list)

print('Retrieved Context:\n')
utils.print_wrapped(relevant_content[:1000] + '...')

Retrieved Context:

deeply integrated into its storytelling and gameplay mechanics. Balancing the benefits
of customization and individualized solutions with concerns about surveillance, manipulation, and
loss of autonomy will be an ongoing challenge  [140]. Our future ability to develop
advanced biometric authentication with standardized protocols will impact our trust in the
technology.
Cyberpunk 2077 challenges the notion of Technological Determinism by presenting a fragmented
future where technological adoption is uneven and often detrimental, reflecting Critical Future
Studies emphasis on the ethical and power dynamics inherent in technological progress  [28], [74].
The games portrayal of uneven technological uptake critiques the idea that technology inherently
leads to societal improvement, underscoring the need for a more nuanced understanding of how
innovations impact different communities. The diverse city districts shown in the game, each one
presenting a very marked own cult

#### 3.2 Generate answer based on retrieved context

In [18]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

answer_prompt_ext = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a detailed answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await llm_acompletion(model="openai/gpt-oss:20b", prompt=answer_prompt_ext)
utils.print_wrapped(answer)

Generated Answer:

**Model used:**
The authors explicitly state that they employed **OpenAI’s GPT‑4** during the preparation of the
manuscript.

**Reason for using GPT‑4:**
- **Readability enhancement:** GPT‑4 was used to polish the prose, making the article clearer and
more accessible to readers.
- **Technical formatting assistance:** The model helped generate correct LaTeX code for references,
tables, and subsections, ensuring that the document complied with the required formatting standards.
- **Efficiency in drafting:** By automating routine writing tasks, GPT‑4 allowed the authors to
focus on higher‑level analysis and synthesis while still maintaining full editorial control over the
final content.

After the model’s output was incorporated, the authors reviewed, edited, and took full
responsibility for the final text, confirming that GPT‑4 served as an assistive tool rather than a
primary author.


## Step 4: Alternative Query

In [19]:
query = "What does a dystopian future look like in this game?"

search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""
tree_search_result = await llm_acompletion(model="openai/gpt-oss:20b", prompt=search_prompt)

In [20]:
print(tree_search_result)

{
    "thinking": "The question asks for a description of the dystopian future depicted in the game. Relevant sections discuss corporate dominance, inequality, surveillance, and the societal impact of advanced technologies. Nodes 0005 and 0006 explicitly describe the game's dystopian setting and its themes. The Discussion (0026) and Conclusion (0027) sections elaborate on the dystopian aspects such as uneven tech adoption, corporate power, and social inequalities. Additionally, node 0013 (Human augmentation and cybernetic enhancements) touches on dystopian elements like cyberpsychosis and corporate exploitation. These nodes collectively provide the answer.",
    "node_list": ["0005", "0006", "0013", "0026", "0027"]
}


In [21]:
tree_search_result_json = json.loads(tree_search_result)
node_map = utils.create_node_mapping(tree_with_text)
print('\nRetrieved Nodes:')
for node_id in tree_search_result_json["node_list"]:
    node = node_map[node_id]
    print(f"Node ID: {node['node_id']}")


Retrieved Nodes:
Node ID: 0005
Node ID: 0006
Node ID: 0013
Node ID: 0026
Node ID: 0027


In [26]:
answer_prompt = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, concise answer based only on the context provided.
"""

answer_prompt_ext = f"""
Answer the question based on the context:

Question: {query}
Context: {relevant_content}

Provide a clear, detailed answer based only on the context provided.
"""

print('Generated Answer:\n')
answer = await llm_acompletion(model="openai/gpt-oss:20b", prompt=answer_prompt_ext)
utils.print_wrapped(answer)

Generated Answer:

**Dystopian Future in *Cyberpunk 2077* (as described in the context)**

The game portrays a future that is far from a single, utopian vision of technology. Instead, it
presents a **fragmented, unevenly‑adopted technological landscape** that amplifies social, cultural,
and ethical tensions. Key elements of this dystopia include:

| Feature | Description | How it contributes to the dystopia |
|---------|-------------|------------------------------------|
| **Uneven tech uptake** | Different districts of Night City adopt and adapt technology in vastly
different ways. | Highlights that progress is not universal; some areas thrive on high‑tech, others
lag, creating stark inequalities. |
| **Corporate dominance & surveillance** | Advanced biometric authentication and pervasive
monitoring are integrated into everyday life. | Corporations wield power over individuals, eroding
privacy and autonomy. |
| **Customization vs manipulation** | Players can tailor their character’s b